# 🏘️ Interpretación de Precios Inmobiliarios (California)

Tras optimizar nuestro modelo de regresión en la Semana 3, ahora vamos a auditar **cómo valora la IA las viviendas**. En este caso, **SHAP** nos revelará el "valor añadido" o "descuento" que cada característica aporta al precio final de la propiedad.

### 💰 ¿Qué buscaremos en este análisis?
1. **Factores Premium:** ¿Qué variables disparan el precio por encima de la media?
2. **Factores de Depreciación:** ¿Qué características restan valor a una vivienda?
3. **Análisis de Ubicación:** Veremos cómo influyen las coordenadas (Latitud/Longitud) en la tasación automática.

---

In [ ]:
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt

# 1. CARGA DEL MODELO V2
path_modelo = '../../../models/modelo_viviendas_optimizado_v2.pkl'
modelo = joblib.load(path_modelo)

# 2. CARGA DE DATOS
path_datos = '../../../data/processed/viviendas_limpio.csv'
df = pd.read_csv(path_datos)

# [Especialista]: Inspeccionamos qué columnas tenemos para evitar el KeyError
print(f"📋 Columnas detectadas en el CSV: {df.columns.tolist()}")

# 3. CREACIÓN DE X (ALINEACIÓN AUTOMÁTICA)
# Extraemos los nombres de las columnas que el modelo espera
columnas_modelo = modelo.feature_names_in_

# [Truco]: Reindexamos el DataFrame completo. 
# Esto solo coge las columnas que el modelo necesita. Si no están, pone 0.
# Si 'MedHouseVal' está en el CSV, simplemente la ignorará al no estar en columnas_modelo.
X = df.reindex(columns=columnas_modelo, fill_value=0)

print(f"✅ X creada con éxito. Forma: {X.shape}")

📋 Columnas detectadas en el CSV: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
✅ X creada con éxito. Forma: (20495, 16)


In [ ]:
# 1. EXPLICADOR PARA REGRESIÓN
# [Explicación]: SHAP analizará cuánto aporta cada variable al precio final en $100,000s.
explainer = shap.TreeExplainer(modelo)

# 2. CÁLCULO DE VALORES SHAP
print("⏳ Calculando el impacto económico de cada variable... Esto abrirá la caja negra inmobiliaria.")
shap_values = explainer.shap_values(X)

# 3. SUMMARY PLOT (Visión Global)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X, show=False)

plt.title("Análisis de Valor: ¿Qué hace que una casa sea cara en California?", fontsize=14)
plt.xlabel("Impacto SHAP (Derecha: Aumenta Precio | Izquierda: Baja Precio)", fontsize=12)
plt.show()